# Class 27: Orchestration with LangGraph

এই notebook সম্পূর্ণ **self-contained** — project folder লাগে না। Section 1-6 LLM ছাড়াই চলে (LangGraph-এর mechanics), Section 7-9 Ollama (`llama3.1:8b`) লাগে।

| Section | কী শিখব |
| --- | --- |
| 1 | State আর reducer — overwrite বনাম append, `InvalidUpdateError` |
| 2 | Conditional edge দিয়ে loop, আর 1.x-এর recursion limit ফাঁদ |
| 3 | `interrupt()` + `Command(resume)` — node আবার চলে! |
| 4 | Checkpointer আর time travel |
| 5 | Streaming mode |
| 6 | Functional API — একই কাজ, অন্য স্টাইল |
| 7 | Mini self-correcting coder, live |
| 8 | Feedback-এর মান: "fail হয়েছে" বনাম failing-test diff |
| 9 | `Send` দিয়ে best-of-N draft (fan-out) |

In [1]:
import json, operator, re, subprocess, sys, tempfile, time
from pathlib import Path
from typing import Annotated, TypedDict

import langgraph
import ollama
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.errors import GraphRecursionError, InvalidUpdateError
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, Send, interrupt

MODEL = "llama3.1:8b"
OPTIONS = {"temperature": 0, "seed": 42, "num_ctx": 8192, "num_predict": 800}
print("langgraph", getattr(langgraph, "__version__", "1.x"))

langgraph 1.x


## 1. State আর Reducer

দুটো key: `last` (reducer নেই → overwrite) আর `log` (`operator.add` → জোড়া লাগে)। দুটো node পরপর লিখলে কী থাকে দেখি।

In [2]:
class S(TypedDict):
    last: str
    log: Annotated[list[str], operator.add]

g = StateGraph(S)
g.add_node("a", lambda s: {"last": "from a", "log": ["a ran"]})
g.add_node("b", lambda s: {"last": "from b", "log": ["b ran"]})
g.add_edge(START, "a"); g.add_edge("a", "b"); g.add_edge("b", END)
print(g.compile().invoke({"last": "", "log": []}))

{'last': 'from b', 'log': ['a ran', 'b ran']}


`last`-এ শুধু শেষ লেখা ("from b"), `log`-এ দুটোই। এবার দুটো node **parallel**-এ (একই super-step) reducer-ছাড়া key-তে লিখলে:

In [3]:
g = StateGraph(S)
g.add_node("a", lambda s: {"last": "from a"})
g.add_node("b", lambda s: {"last": "from b"})
g.add_edge(START, "a"); g.add_edge(START, "b")          # fan-out: a আর b একসাথে
g.add_edge("a", END); g.add_edge("b", END)
try:
    g.compile().invoke({"last": "", "log": []})
except InvalidUpdateError as e:
    print("InvalidUpdateError:", str(e).splitlines()[0])

InvalidUpdateError: At key 'last': Can receive only one value per step. Use an Annotated key to handle multiple values.


## 2. Loop = Conditional Edge, আর থামার শর্ত

একটা "fake" fix loop: প্রতি attempt-এ `attempts` বাড়ে, ৩-এ পৌঁছালে "pass"। থামার শর্ত router-এ, state থেকে।

In [4]:
class L(TypedDict):
    attempts: int
    history: Annotated[list[str], operator.add]

def attempt(s):
    n = s["attempts"] + 1
    return {"attempts": n, "history": [f"attempt {n}: {'PASS' if n >= 3 else 'FAIL'}"]}

def route(s):
    return END if s["attempts"] >= 3 else "attempt"

g = StateGraph(L)
g.add_node("attempt", attempt)
g.add_edge(START, "attempt")
g.add_conditional_edges("attempt", route, ["attempt", END])
print(g.compile().invoke({"attempts": 0, "history": []})["history"])

['attempt 1: FAIL', 'attempt 2: FAIL', 'attempt 3: PASS']


### Recursion limit ফাঁদ

এবার router কখনো থামে না (একটা bug)। ২০২৪-এর tutorial বলে default limit ২৫ — তাই loop নিজেই থামবে। **LangGraph 1.0.6 থেকে default ~১০ হাজার।** llama3.1:8b-এর প্রতি call ~১৫-২০ সেকেন্ড হলে সেটা ঘণ্টার পর ঘণ্টা। তাই limit নিজে দিন:

In [5]:
g = StateGraph(L)
g.add_node("attempt", attempt)
g.add_edge(START, "attempt")
g.add_conditional_edges("attempt", lambda s: "attempt", ["attempt"])   # bug: never stops
loop = g.compile()
try:
    loop.invoke({"attempts": 0, "history": []}, {"recursion_limit": 12})
except GraphRecursionError as e:
    print("GraphRecursionError:", str(e).splitlines()[0][:110])

GraphRecursionError: Recursion limit of 12 reached without hitting a stop condition. You can increase the limit by setting the `rec


## 3. `interrupt()` — resume-এ node আবার চলে

Approval node-এ `interrupt()`-এর আগে একটা counter বাড়াই। একবার pause, একবার resume — counter কত হয়?

In [6]:
calls = {"before_interrupt": 0}

class A(TypedDict):
    code: str
    approved: bool

def approval(s):
    calls["before_interrupt"] += 1                     # imagine an expensive LLM call / test run here
    answer = interrupt({"question": "approve?", "code": s["code"]})
    return {"approved": answer == "yes"}

g = StateGraph(A)
g.add_node("approval", approval)
g.add_edge(START, "approval"); g.add_edge("approval", END)
app = g.compile(checkpointer=InMemorySaver())          # interrupt needs a checkpointer + thread_id
cfg = {"configurable": {"thread_id": "t1"}}

first = app.invoke({"code": "def f(): ...", "approved": False}, cfg)
print("paused with:", first["__interrupt__"][0].value)
print("next node:", app.get_state(cfg).next)
final = app.invoke(Command(resume="yes"), cfg)
print("approved:", final["approved"], "| code before interrupt ran", calls["before_interrupt"], "times")

paused with: {'question': 'approve?', 'code': 'def f(): ...'}
next node: ('approval',)
approved: True | code before interrupt ran 2 times


দুইবার! তাই নিয়ম: **interrupt-এর আগে কোনো দামি বা side-effect কাজ না** — approval-কে নিজের node দিন।

## 4. Checkpointer আর Time Travel

Section 2-এর loop একটা checkpointer দিয়ে চালাই, তারপর প্রতিটা checkpoint দেখি আর attempt ১-এর পরের অবস্থা থেকে replay করি।

In [7]:
g = StateGraph(L)
g.add_node("attempt", attempt)
g.add_edge(START, "attempt")
g.add_conditional_edges("attempt", route, ["attempt", END])
app = g.compile(checkpointer=InMemorySaver())
cfg = {"configurable": {"thread_id": "tt"}}
app.invoke({"attempts": 0, "history": []}, cfg)

history = list(app.get_state_history(cfg))            # newest first
for snap in history:
    print(f"step {snap.metadata['step']:>2}  next={snap.next or ('END',)}  attempts={snap.values.get('attempts')}")

step  3  next=('END',)  attempts=3
step  2  next=('attempt',)  attempts=2
step  1  next=('attempt',)  attempts=1
step  0  next=('attempt',)  attempts=0
step -1  next=('__start__',)  attempts=None


In [8]:
after_first = next(s for s in history if s.values.get("attempts") == 1)
replayed = app.invoke(None, after_first.config)        # nodes re-execute from that point
print("replay from attempts=1 ->", replayed["history"])
forked = app.update_state(after_first.config, {"attempts": 2})   # fork: pretend attempt 2 already happened
print("fork ->", app.invoke(None, forked)["history"])

replay from attempts=1 -> ['attempt 1: FAIL', 'attempt 2: FAIL', 'attempt 3: PASS']
fork -> ['attempt 1: FAIL', 'attempt 3: PASS']


## 5. Streaming

`updates` = প্রতিটা node-এর delta; `values` = প্রতি ধাপের পুরো state। UI-তে progress দেখাতে সাধারণত `updates`।

In [9]:
app = g.compile()
print("stream_mode='updates':")
for chunk in app.stream({"attempts": 0, "history": []}, stream_mode="updates"):
    print("  ", chunk)
print("stream_mode='values' (last 2):")
print("  ", list(app.stream({"attempts": 0, "history": []}, stream_mode="values"))[-2:])

stream_mode='updates':
   {'attempt': {'attempts': 1, 'history': ['attempt 1: FAIL']}}
   {'attempt': {'attempts': 2, 'history': ['attempt 2: FAIL']}}
   {'attempt': {'attempts': 3, 'history': ['attempt 3: PASS']}}
stream_mode='values' (last 2):
   [{'attempts': 2, 'history': ['attempt 1: FAIL', 'attempt 2: FAIL']}, {'attempts': 3, 'history': ['attempt 1: FAIL', 'attempt 2: FAIL', 'attempt 3: PASS']}]


## 6. Functional API — একই loop, সাধারণ Python

`@entrypoint` + `@task`। পার্থক্য: resume-এ entrypoint শুরু থেকে চলে, কিন্তু শেষ হওয়া `@task`-এর ফল checkpoint থেকে আসে — আবার চলে না। দাম: graph আঁকা যায় না।

In [10]:
from langgraph.func import entrypoint, task

runs = {"run_tests": 0}

@task
def run_tests(code: str) -> bool:
    runs["run_tests"] += 1
    return "return a + b" in code

@entrypoint(checkpointer=InMemorySaver())
def review(code: str) -> str:
    ok = run_tests(code).result()
    answer = interrupt({"tests_passed": ok})          # pause for a human
    return f"tests_passed={ok}, human said {answer}"

cfg = {"configurable": {"thread_id": "fn"}}
review.invoke("def add(a, b):\n    return a + b", cfg)
print(review.invoke(Command(resume="ship it"), cfg))
print("run_tests executed", runs["run_tests"], "time(s) across pause + resume")

tests_passed=True, human said ship it
run_tests executed 1 time(s) across pause + resume


## 7. Mini Self-Correcting Coder — live

Project-এর ছোট সংস্করণ: (১) model code লেখে, (২) আলাদা process-এ প্রতিটা test চলে, (৩) fail হলে failing-test diff সহ retry, (৪) budget শেষ বা pass। সব repair step temperature 0 — Self-Debugging paper-এর মতো greedy।

In [11]:
FENCE = re.compile(r"```(?:python)?\s*\n(.*?)```", re.S)

def generate(prompt: str) -> str:
    r = ollama.chat(model=MODEL, options=OPTIONS, messages=[
        {"role": "system", "content": "Write one self-contained Python function. Reply with one ```python block only."},
        {"role": "user", "content": prompt}])
    blocks = FENCE.findall(r.message.content)
    return (blocks[-1] if blocks else r.message.content).strip()

HARNESS = '''
import json, runpy
ns = runpy.run_path("solution.py")
out = []
for call, expected in json.load(open("tests.json")):
    try:
        got = eval(call, dict(ns)); want = eval(expected)
        out.append([call, got == want and type(got) == type(want), f"returned {got!r}, expected {want!r}"])
    except BaseException as e:
        out.append([call, False, f"raised {type(e).__name__}: {e}"])
print("@@" + json.dumps(out))
'''

def run_tests(code: str, tests) -> list:
    with tempfile.TemporaryDirectory() as d:
        Path(d, "solution.py").write_text(code); Path(d, "tests.json").write_text(json.dumps(tests))
        Path(d, "h.py").write_text(HARNESS)
        try:
            p = subprocess.run([sys.executable, "-I", "h.py"], cwd=d, capture_output=True, text=True,
                               timeout=10, env={"PATH": "/usr/bin:/bin"})
        except subprocess.TimeoutExpired:
            return [[c, False, "timeout"] for c, _ in tests]
    if "@@" not in p.stdout:
        return [[c, False, (p.stderr.strip().splitlines() or ['crash'])[-1]] for c, _ in tests]
    return json.loads(p.stdout.split("@@", 1)[1])

print(run_tests("def add(a, b):\n    return a - b", [["add(2, 3)", "5"], ["add(0, 0)", "0"]]))

[['add(2, 3)', False, 'returned -1, expected 5'], ['add(0, 0)', True, 'returned 0, expected 0']]


In [12]:
class Coder(TypedDict, total=False):
    task: str
    tests: list
    code: str
    attempts: int
    feedback: str
    passed: bool
    history: Annotated[list[str], operator.add]

def build_coder(detailed_feedback: bool = True, max_attempts: int = 4):
    def write(s):
        prompt = s["task"] + "\nExamples: " + "; ".join(f"{c} -> {e}" for c, e in s["tests"])
        if s.get("feedback"):
            prompt += f"\n\nYour previous code:\n```python\n{s['code']}\n```\nIt failed:\n{s['feedback']}\nFix it."
        n = s.get("attempts", 0) + 1
        return {"code": generate(prompt), "attempts": n}

    def test(s):
        res = run_tests(s["code"], s["tests"])
        fails = [r for r in res if not r[1]]
        if detailed_feedback:
            fb = "\n".join(f"- {c}: {msg}" for c, _, msg in fails)
        else:
            fb = "Some tests failed."                       # Section 8 compares this with the detailed version
        return {"passed": not fails, "feedback": fb,
                "history": [f"attempt {s['attempts']}: {len(res) - len(fails)}/{len(res)} tests passed"]}

    def route(s):
        if s["passed"]:
            return END
        return "write" if s["attempts"] < max_attempts else END

    g = StateGraph(Coder)
    g.add_node("write", write); g.add_node("test", test)
    g.add_edge(START, "write"); g.add_edge("write", "test")
    g.add_conditional_edges("test", route, ["write", END])
    return g.compile()

TASK = {"task": ("Write format_duration(seconds: int) -> str that formats a non-negative number of seconds in years "
                 "(365 days), days, hours, minutes and seconds, largest first, skipping zero units, with correct "
                 "singular/plural. Join with ', ' except the last two parts, which are joined with ' and '. 0 returns 'now'."),
        "tests": [["format_duration(62)", "'1 minute and 2 seconds'"],
                  ["format_duration(3662)", "'1 hour, 1 minute and 2 seconds'"],
                  ["format_duration(0)", "'now'"],
                  ["format_duration(1)", "'1 second'"],
                  ["format_duration(31536000 + 2 * 86400 + 5)", "'1 year, 2 days and 5 seconds'"]]}

t0 = time.time()
result = build_coder().invoke(TASK, {"recursion_limit": 30})
print("\n".join(result["history"]), f"\npassed={result['passed']} in {time.time() - t0:.0f}s\n")
print(result["code"])

attempt 1: 3/5 tests passed
attempt 2: 5/5 tests passed 
passed=True in 94s

def format_duration(seconds: int) -> str:
    if seconds == 0:
        return 'now'

    years, seconds = divmod(seconds, 31536000)
    days, seconds = divmod(seconds, 86400)
    hours, seconds = divmod(seconds, 3600)
    minutes, seconds = divmod(seconds, 60)

    units = []
    if years:
        units.append(f'{years} year{"s" if years > 1 else ""}')
    if days:
        units.append(f'{days} day{"s" if days > 1 else ""}')
    if hours:
        units.append(f'{hours} hour{"s" if hours > 1 else ""}')
    if minutes:
        units.append(f'{minutes} minute{"s" if minutes > 1 else ""}')
    if seconds:
        units.append(f'{seconds} second{"s" if seconds > 1 else ""}')

    if len(units) == 1:
        return units[0]
    elif len(units) == 2:
        return ' and '.join(units)
    else:
        return ', '.join(units[:-2]) + ', ' + ' and '.join(units[-2:])


## 8. Feedback-এর মান

"Is Self-Repair a Silver Bullet?" বলে feedback-ই bottleneck। একই task, দুই রকম feedback: শুধু "Some tests failed" বনাম প্রতিটা failing test-এর expected/actual।

In [13]:
for detailed in (False, True):
    t0 = time.time()
    r = build_coder(detailed_feedback=detailed).invoke(TASK, {"recursion_limit": 30})
    label = "failing-test diff" if detailed else "'Some tests failed.'"
    print(f"{label:<22} -> passed={r['passed']} after {r['attempts']} attempt(s), {time.time() - t0:.0f}s | {r['history']}")

'Some tests failed.'   -> passed=True after 2 attempt(s), 117s | ['attempt 1: 3/5 tests passed', 'attempt 2: 5/5 tests passed']


failing-test diff      -> passed=True after 2 attempt(s), 111s | ['attempt 1: 3/5 tests passed', 'attempt 2: 5/5 tests passed']


**আমাদের run-এ:** এই একটা task-এ দুই রকম feedback-ই ২য় attempt-এ পাস করেছে — পার্থক্য দেখা যায়নি। এক task থেকে সিদ্ধান্ত হয় না: project-এর ১১-task benchmark-এ `word_freq` generic hint পেয়ে আরও খারাপ হয়েছিল। Feedback-এর প্রভাব মাপতে অনেক task আর একাধিক run লাগে।

## 9. `Send` — best-of-N draft (fan-out)

Olausson et al.: compute সমান রাখলে অনেক repair-এর চেয়ে বেশি স্বাধীন draft প্রায়ই সমান বা ভালো। `Send` দিয়ে ৩টা draft (temperature 0.8, আলাদা seed) — reducer-এ জমা হয়, তারপর সেরাটা বাছা হয়।

In [14]:
class Best(TypedDict, total=False):
    task: str
    tests: list
    drafts: Annotated[list[dict], operator.add]

def fan_out(s):
    return [Send("draft", {"task": s["task"], "tests": s["tests"], "seed": seed}) for seed in (1, 2, 3)]

def draft(p):
    prompt = p["task"] + "\nExamples: " + "; ".join(f"{c} -> {e}" for c, e in p["tests"])
    r = ollama.chat(model=MODEL, options={**OPTIONS, "temperature": 0.8, "seed": p["seed"]},
                    messages=[{"role": "system", "content": "Write one Python function in one ```python block."},
                              {"role": "user", "content": prompt}])
    blocks = FENCE.findall(r.message.content)
    code = (blocks[-1] if blocks else r.message.content).strip()
    score = sum(ok for _, ok, _ in run_tests(code, p["tests"]))
    return {"drafts": [{"seed": p["seed"], "score": score, "code": code}]}

g = StateGraph(Best)
g.add_node("draft", draft)
g.add_conditional_edges(START, fan_out, ["draft"])
g.add_edge("draft", END)
best = g.compile().invoke({**TASK, "drafts": []})
for d in sorted(best["drafts"], key=lambda d: -d["score"]):
    print(f"seed {d['seed']}: {d['score']}/{len(TASK['tests'])} tests")

seed 2: 4/5 tests
seed 3: 3/5 tests
seed 1: 2/5 tests


**আমাদের run-এ:** তিনটা স্বাধীন draft-এর কোনোটাই ৫/৫ পাস করেনি (সেরা ৪/৫), অথচ Section 7-এর repair loop ২য় attempt-এ ৫/৫। এই task-এ নির্দিষ্ট feedback-সহ repair, অন্ধ sampling-কে হারিয়েছে — Olausson-এর "sampling প্রায়ই সমান বা ভালো" কথাটা গড়ের হিসাব, প্রতিটা task-এ সত্য না। সিদ্ধান্ত নিতে নিজের task set-এ সমান call budget-এ তুলনা করুন (Brain Teaser ৪)।

In [15]:
print(build_coder().get_graph().draw_mermaid())   # text only; draw_mermaid_png() would call the mermaid.ink web service

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	write(write)
	test(test)
	__end__([<p>__end__</p>]):::last
	__start__ --> write;
	test -.-> __end__;
	test -.-> write;
	write --> test;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 🧠 Brain Teasers

1. Section 3-এ approval-কে দুটো node-এ ভাগ করুন (`prepare` → `approval`) যাতে counter একবারই বাড়ে। কেন কাজ করে?
2. Section 4-এর fork-এ `{"attempts": 2}`-এর বদলে `{"history": ["manual note"]}` দিলে history-তে কী হয়? (Hint: reducer।)
3. Section 8 কয়েকবার চালান (temperature 0)। ফল কি বদলায়? কেন বদলাতে পারে, যদিও seed স্থির?
4. Section 9-এর best-of-3 আর Section 7-এর repair loop-এর LLM call সংখ্যা সমান করে (দুটোই ৩) তুলনা করুন — কোনটা জেতে?